In [ ]:
from pyspark.sql import Row
from pyspark.sql import sparkSession

spark = SparkSession.builder.appName("SCD_1_Implementation").getOrCreate()

# Create a tiny test DataFrame
new_data = [Row(emp_id=6, name="TestUser", salary=55000)]
new_df = spark.createDataFrame(new_data)

# Write into MySQL (append mode)
new_df.write.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", TABLE) \
    .option("user", USER) \
    .option("password", PASSWORD) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .mode("append") \
    .save()

print("✅ Inserted TestUser into target_employees")

AttributeError: 'NoneType' object has no attribute 'sc'

In [4]:
from pyspark.sql import SparkSession

# Stop any existing Spark session
try:
    spark.stop()
except:
    pass

# Create SparkSession with MySQL connector auto-downloaded
spark = SparkSession.builder \
    .appName("mysql_test") \
    .config("spark.jars.packages", "com.mysql:mysql-connector-j:8.0.33") \
    .getOrCreate()

# MySQL connection parameters
HOST = "3.134.89.129"
USER = "root"
PASSWORD = "Yashwant!14"
DATABASE = "practice_db"
TABLE = "target_employees"

jdbc_url = f"jdbc:mysql://{HOST}:3306/{DATABASE}?useSSL=false"

# Read table from MySQL
df = spark.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", TABLE) \
    .option("user", USER) \
    .option("password", PASSWORD) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()

print("✅ Connection successful, here are some rows:")
df.show()


from pyspark.sql import Row

# Create a tiny test DataFrame
new_data = [Row(emp_id=6, name="TestUser", salary=55000)]
new_df = spark.createDataFrame(new_data)

# Write into MySQL (append mode)
new_df.write.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", TABLE) \
    .option("user", USER) \
    .option("password", PASSWORD) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .mode("append") \
    .save()

print("✅ Inserted TestUser into target_employees")

# Stop Spark
spark.stop()

✅ Connection successful, here are some rows:


+------+-------+------+
|emp_id|   name|salary|
+------+-------+------+
|     1|   Yash| 52000|
|     2| Mukesh| 62000|
|     3|    Ram| 72000|
|     4|Krishna| 82000|
+------+-------+------+



✅ Inserted TestUser into target_employees


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Initialize Spark session


# Kill any existing session
try:
    spark.stop()
except:
    pass

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("mysql_test") \
    .config("spark.jars", "/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/jars/mysql-connector-j-8.0.33.jar") \
    .config("spark.driver.extraClassPath", "/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/jars/mysql-connector-j-8.0.33.jar") \
    .config("spark.executor.extraClassPath", "/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/jars/mysql-connector-j-8.0.33.jar") \
    .getOrCreate()

# MySQL connection parameters
HOST = "3.134.89.129"
USER = "root"
PASSWORD = "Yashwant!14"
DATABASE = "practice-db"
TABLE = "target_employees"

jdbc_url = f"jdbc:mysql://{HOST}:3306/{DATABASE}?useSSL=false"

# Read target table from MySQL
data_target_df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", TABLE) \
    .option("user", USER) \
    .option("password", PASSWORD) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()

print("Target Table (from MySQL):")
data_target_df.show()




'''
data_target = [    
    (1, "Yash", 52000),
    (2, "Mukesh", 62000),
    (3, "Ram", 72000),
    (4, "Krishna", 82000),

]

data_target_df = spark.createDataFrame(data_target, schema)
'''



print("Target Table:")
data_target_df.show()



schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("salary", IntegerType(), True)])



# New data to be merged

data_new = [    
    (1, "Yash", 52000), #no change
    (2, "Mukesh", 60000), #salary changed
    (3, "Charlie", 70000), #name changed
    (5, "Radha", 90000), #new record
]   
data_new_df = spark.createDataFrame(data_new, schema)

print("New Data:")
data_new_df.show()


#----------------------------------------------------------------------------------------------------
#changed
changed_condition = ((data_new_df.name != data_target_df.name) | \
            (data_new_df.salary != data_target_df.salary))

changed_records_df = data_new_df.join(data_target_df, on='emp_id', how='inner') \
    .where(changed_condition) \
    .select(data_new_df.emp_id, data_new_df.name, data_new_df.salary)

print("Changed Records:")
changed_records_df.show(truncate=False)
print("Count of Changed Records:", changed_records_df.count())
#----------------------------------------------------------------------------------------------------



#----------------------------------------------------------------------------------------------------
#new
new_records_df = data_new_df.join(data_target_df, on='emp_id', how='left_anti')

print("New Records:")
new_records_df.show(truncate=False)
#----------------------------------------------------------------------------------------------------

#----------------------------------------------------------------------------------------------------
#history
history_records_df = data_target_df.join(data_new_df, on='emp_id', how='left_anti')  

print("history Records:")
history_records_df.show(truncate=False)
#----------------------------------------------------------------------------------------------------


#----------------------------------------------------------------------------------------------------
#unchanged
unchanged_condition = ((data_new_df.name == data_target_df.name) & \
            (data_new_df.salary == data_target_df.salary))

unchanged_records_df = data_new_df.join(data_target_df, on='emp_id', how='inner') \
    .where(unchanged_condition) \
    .select(data_new_df.emp_id, data_new_df.name, data_new_df.salary)

print("Unchanged Records:")
unchanged_records_df.show(truncate=False)
#----------------------------------------------------------------------------------------------------



final_df = history_records_df.union(changed_records_df).union(new_records_df).union(unchanged_records_df).orderBy("emp_id")

print("Final DataFrame:")
final_df.show(truncate=False)

spark.stop()




Py4JJavaError: An error occurred while calling o54.load.
: java.lang.ClassNotFoundException: com.mysql.cj.jdbc.Driver
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:476)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:594)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:527)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register(DriverRegistry.scala:46)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1$adapted(JDBCOptions.scala:103)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:41)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcRelationProvider.createRelation(JdbcRelationProvider.scal